# 數位控制系統第一章：導論與系統建模

本筆記本整合了 **理論推導** 與 **MATLAB/Octave 程式碼**，讓你在同一份文件中同時理解公式與程式實作。

---

## 🔧 環境設定

在 Octave Jupyter Notebook 中繪圖，需要：
1. 使用 `%plot -f svg` 或 `graphics_toolkit('gnuplot')` 讓圖表能內嵌顯示
2. 載入 `control` 套件（提供 `tf`, `ss`, `step` 等控制工具函數）
3. 設定中文字型以避免亂碼

In [ ]:
% 抑制圖形引擎切換時的警告訊息
warning('off', 'Octave:graphics-toolkit');
% 使用 gnuplot 引擎，確保 Jupyter 內嵌繪圖正常運作
graphics_toolkit('gnuplot');
clear; clc;
pkg load control;

% 設定字體以支援中文顯示
set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

---

## 📖 核心函數介紹：`tf()` 與 `ss()`

在 MATLAB / Octave 的 **Control System Toolbox** 中，最常用的兩個建模函數是 `tf()` 和 `ss()`。

### `tf(num, den)` — 建立轉移函數模型

**語法：** `G = tf(分子係數向量, 分母係數向量)`

轉移函數的一般形式：

$$G(s) = \frac{b_m s^m + b_{m-1} s^{m-1} + \cdots + b_1 s + b_0}{a_n s^n + a_{n-1} s^{n-1} + \cdots + a_1 s + a_0}$$

`num` 和 `den` 就是把 **多項式的係數由高次排到低次** 放進一個向量（陣列）裡。

| 範例 | 數學式 | MATLAB 寫法 | 說明 |
|------|--------|-------------|------|
| 純增益 | $G(s) = 5$ | `tf([5], [1])` | 分子 = 5，分母 = 1 |
| 一階 | $G(s) = \frac{1}{s+2}$ | `tf([1], [1 2])` | 分母 $s+2$ 的係數 = `[1 2]` |
| 二階 | $G(s) = \frac{1}{0.6 s^2}$ | `tf([1], [0.6 0 0])` | 分母 $0.6s^2 + 0s + 0$ → `[0.6 0 0]` |
| 二階阻尼 | $G(s) = \frac{10}{0.5s^2 + 0.1s + 10}$ | `tf([10], [0.5 0.1 10])` | 分母 = $0.5s^2 + 0.1s + 10$ |

#### 🔑 關鍵概念：`[J 0 0]` 代表什麼？

以衛星模型為例：

$$G_p(s) = \frac{1}{Js^2} = \frac{1}{Js^2 + 0 \cdot s + 0}$$

把分母 $Js^2 + 0s + 0$ 的**每一項係數**由高次到低次排列 → `[J 0 0]`

- 第 1 個 `J`：$s^2$ 的係數
- 第 2 個 `0`：$s^1$ 的係數（沒有 $s$ 項 → 0）
- 第 3 個 `0`：$s^0$（常數項）的係數（也是 0）

> **白話文：向量裡有幾個元素，就代表多項式最高是幾次（元素數 - 1 = 最高次數）。**
> `[J 0 0]` 有 3 個元素 → 最高次是 $s^2$ → 對應 $Js^2$。

---

### `ss(A, B, C, D)` — 建立狀態空間模型

**語法：** `sys = ss(A, B, C, D)`

狀態空間模型的標準形式：

$$\dot{\mathbf{x}}(t) = A \mathbf{x}(t) + B u(t)$$
$$y(t) = C \mathbf{x}(t) + D u(t)$$

| 參數 | 意義 | 維度（$n$ 狀態、$m$ 輸入、$p$ 輸出） |
|------|------|------|
| `A` | 系統矩陣（描述狀態之間的耦合關係） | $n \times n$ |
| `B` | 輸入矩陣（輸入如何影響狀態） | $n \times m$ |
| `C` | 輸出矩陣（哪些狀態被量測輸出） | $p \times n$ |
| `D` | 直饋矩陣（輸入直接影響輸出，通常為 0） | $p \times m$ |

> **何時用 `tf`？何時用 `ss`？**
> - 系統可以寫成簡單的 $\frac{分子}{分母}$ 多項式比 → 用 `tf()`
> - 系統從物理方程式推導出矩陣形式 → 用 `ss()`
> - 兩者可互轉：`tf(sys)` 把 ss 轉成 tf，`ss(G)` 把 tf 轉成 ss

---

### 其他常用函數

| 函數 | 功能 | 範例 |
|------|------|------|
| `step(sys)` | 繪製單位階躍響應（輸入 = 從 0 跳到 1） | `step(Gp_sat)` |
| `step(sys, t)` | 指定時間向量的階躍響應 | `step(Gp_sat, 0:0.1:50)` |
| `[y, t] = step(sys)` | 取得輸出數據而非直接畫圖 | 用來自訂繪圖 |
| `impulse(sys)` | 繪製脈衝響應 | `impulse(Gp_sat)` |
| `bode(sys)` | 繪製波德圖（頻率響應） | `bode(Gp_sat)` |
| `pole(sys)` | 求系統極點 | `pole(Gp_smib)` |
| `zero(sys)` | 求系統零點 | `zero(Gp_smib)` |

---

## 一、衛星系統建模 (Satellite Model)

### 物理方程式

球形衛星繞航向軸（Yaw-axis）旋轉，推力器產生轉矩 $v(t)$ 來控制角度 $\theta(t)$。

根據牛頓旋轉定律：

$$J \frac{d^2 \theta(t)}{dt^2} = v(t)$$

其中 $J$ 為繞航向軸的轉動慣量（$\text{kg} \cdot \text{m}^2$），$v(t)$ 為推力器力矩（$\text{N} \cdot \text{m}$）。

### 轉移函數

拉普拉斯轉換後（忽略初始條件）：

$$Js^2 \Theta(s) = V(s) \quad \Rightarrow \quad G_p(s) = \frac{\Theta(s)}{V(s)} = \frac{1}{Js^2}$$

### 狀態空間模型

定義 $x_1 = \theta$，$x_2 = \dot{\theta}$：

$$\dot{\mathbf{x}} = \begin{bmatrix} 0 & 1 \\ 0 & 0 \end{bmatrix} \mathbf{x} + \begin{bmatrix} 0 \\ \frac{1}{J} \end{bmatrix} v$$

> 💡 **白話文**：衛星在太空中幾乎無摩擦，一推就停不下來。數學上分母只有 $s^2$（兩個積分器），
> 給一個固定力矩，角度會像拋物線一樣越來越快地偏移。

### 程式碼對照

```
J = 0.6;
Gp_sat = tf([1], [J 0 0]);
```

| 程式碼 | 數學對應 |
|--------|----------|
| `J = 0.6` | 轉動慣量 $J = 0.6 \text{ kg}\cdot\text{m}^2$ |
| `[1]` | 分子 = 1 |
| `[J 0 0]` = `[0.6 0 0]` | 分母 = $0.6s^2 + 0s + 0 = 0.6s^2$ |
| 整體 | $G_p(s) = \frac{1}{0.6s^2}$ |

In [ ]:
% === 衛星系統 ===
% 物理意義：J = 轉動慣量 (kg·m²)
J = 0.6;

% tf([分子係數], [分母係數])
% 分子 = 1              → [1]
% 分母 = J*s^2 + 0*s + 0 → [J 0 0] = [0.6 0 0]
% 整體 = 1 / (0.6 s^2)
Gp_sat = tf([1], [J 0 0]);

% 印出轉移函數，驗證是否正確
disp('=== 衛星轉移函數 ===');
Gp_sat

---

## 二、直流伺服馬達系統建模 (DC Servo Motor)

### 物理方程式

直流馬達（電樞控制、磁場恆定、忽略電感 $L_a$）：

- 電路方程：$e(t) = i(t) R_a + K_b \dot{\theta}(t)$
- 電磁轉矩：$v(t) = K_T i(t)$
- 機械平衡：$v(t) = J \ddot{\theta}(t) + B \dot{\theta}(t)$

### 轉移函數

$$G_p(s) = \frac{\Theta(s)}{E(s)} = \frac{K_T / (JR_a)}{s\left(s + \frac{BR_a + K_T K_b}{JR_a}\right)}$$

### 狀態空間模型

定義 $x_1 = \theta$，$x_2 = \dot{\theta}$：

$$A = \begin{bmatrix} 0 & 1 \\ 0 & -\frac{B R_a + K_T K_b}{J R_a} \end{bmatrix}, \quad B = \begin{bmatrix} 0 \\ \frac{K_T}{J R_a} \end{bmatrix}, \quad C = \begin{bmatrix} 1 & 0 \end{bmatrix}, \quad D = 0$$

> 💡 **白話文**：馬達比衛星多了一個「摩擦阻尼 + 反電動勢」的減速機制。
> 所以給定電壓後，轉速會慢慢穩定在一個值（不像衛星會永遠加速）。
> 但角度仍然是持續增加的（分母有一個 $s$ 代表一個積分器）。

### 程式碼對照

這裡因為公式比較複雜，直接使用 **狀態空間 `ss()` 建模** 而非 `tf()`。

```
A22 = -(Bm * Ra + KT * Kb) / (Jm * Ra);
B2  = KT / (Jm * Ra);
sys_motor = ss([0 1; 0 A22], [0; B2], [1 0], 0);
```

| 程式碼 | 數學對應 |
|--------|----------|
| `[0 1; 0 A22]` | 系統矩陣 $A = \begin{bmatrix} 0 & 1 \\ 0 & A_{22} \end{bmatrix}$ |
| `[0; B2]` | 輸入矩陣 $B = \begin{bmatrix} 0 \\ B_2 \end{bmatrix}$ |
| `[1 0]` | 輸出矩陣 $C$：只量測角度 $x_1 = \theta$ |
| `0` | 直饋矩陣 $D = 0$（輸入不直接出現在輸出）|

In [ ]:
% === 直流伺服馬達 ===
% 各物理參數
Ra = 2;      % 電樞電阻 (Ω)
Kb = 0.1;    % 反電動勢常數 (V·s/rad)
KT = 0.1;    % 電磁轉矩常數 (N·m/A)
Jm = 0.05;   % 馬達轉動慣量 (kg·m²)
Bm = 0.01;   % 黏滯摩擦係數 (N·m·s/rad)

% 計算狀態空間矩陣 A 的 (2,2) 元素
% A22 = -(B*Ra + KT*Kb) / (J*Ra)
%     = -(0.01*2 + 0.1*0.1) / (0.05*2)
%     = -0.03 / 0.1 = -0.3
A22 = -(Bm * Ra + KT * Kb) / (Jm * Ra);

% 計算輸入矩陣 B 的第 2 個元素
% B2 = KT / (J*Ra) = 0.1 / (0.05*2) = 1.0
B2 = KT / (Jm * Ra);

% 用 ss() 建立狀態空間模型
% A = [0  1;  0  A22]    狀態如何影響狀態
% B = [0; B2]            輸入如何影響狀態
% C = [1 0]              輸出 = x1 = 角度 θ
% D = 0                  沒有直饋
sys_motor = ss([0 1; 0 A22], [0; B2], [1 0], 0);

disp('=== 馬達狀態空間模型 ===');
sys_motor

% 也可以轉成轉移函數來看
disp('=== 馬達轉移函數 (由 ss 轉換) ===');
tf(sys_motor)

---

## 三、SMIB 電力系統建模 (Single Machine Infinite Bus)

### 物理方程式

同步發電機的擺動方程（小信號線性化後）：

$$G_p(s) = \frac{k}{Ms^2 + ds + k}$$

其中：
- $M$：發電機角動量
- $d$：阻尼係數
- $k = \frac{E \cos \delta_0}{x}$：同步力係數（電網「彈簧」的勁度）

### 狀態空間模型

$$\dot{\mathbf{x}} = \begin{bmatrix} 0 & 1 \\ -\frac{k}{M} & -\frac{d}{M} \end{bmatrix} \mathbf{x} + \begin{bmatrix} 0 \\ \frac{1}{M} \end{bmatrix} \Delta P_m$$

> 💡 **白話文**：這是標準的 **二階彈簧-質量-阻尼** 系統，分母三個係數分別對應慣性、阻尼、彈簧。
> 因為分母所有係數都是正的，系統是穩定的，階躍響應會震盪後收斂到穩態值。

### 程式碼對照

```
M = 0.5; d = 0.1; k = 10;
Gp_smib = tf([k], [M d k]);
```

| 程式碼 | 數學對應 |
|--------|----------|
| `[k]` = `[10]` | 分子 = $k = 10$ |
| `[M d k]` = `[0.5 0.1 10]` | 分母 = $0.5s^2 + 0.1s + 10$ |
| 整體 | $G_p(s) = \frac{10}{0.5s^2 + 0.1s + 10}$ |

> 注意：分母的三個位置分別是 $s^2$、$s^1$、$s^0$ 的係數，
> 對應物理的「慣性」「阻尼」「彈簧」。

In [ ]:
% === SMIB 電力系統 ===
M = 0.5;   % 角動量 (代表發電機轉子的慣性)
d = 0.1;   % 阻尼係數
k = 10;    % 同步力係數 (電網的「彈簧勁度」)

% tf([分子], [分母])
% 分子 = k = 10                        → [10]
% 分母 = M*s^2 + d*s + k = 0.5s²+0.1s+10 → [0.5 0.1 10]
Gp_smib = tf([k], [M d k]);

disp('=== SMIB 轉移函數 ===');
Gp_smib

% 順便算一下極點，看看系統是否穩定
disp('=== SMIB 極點 ===');
pole(Gp_smib)

---

## 四、溫控艙系統建模 (Temperature Control System)

### 物理方程式

恆溫水槽的能量守恆：供應熱量 = 儲存熱量 + 散失熱量

線性化後得到 **一階滯後模型**：

$$G_p(s) = \frac{K}{\tau s + 1}$$

其中：
- $K = \frac{1}{VH + 1/R}$：穩態增益
- $\tau = \frac{C}{VH + 1/R}$：時間常數（$C$ = 熱容量，$R$ = 熱阻）

### 狀態空間模型 （純量形式）

$$\dot{x}(t) = -\frac{VH + 1/R}{C} \cdot x(t) + \frac{1}{C} \cdot u(t)$$

> 💡 **白話文**：溫控系統是最單純的一階系統。只有一個「能量儲存元件」（水的熱容）。
> 加熱後溫度像爬坡一樣平滑上升，不會震盪。時間常數 $\tau$ 越大，爬得越慢。

### 程式碼對照

```
C_thermal = 100; R_thermal = 2; VH = 0.5;
tau  = C_thermal / (VH + 1/R_thermal);
Gain = 1 / (VH + 1/R_thermal);
Gp_temp = tf([Gain], [tau 1]);
```

| 程式碼 | 數學對應 |
|--------|----------|
| `VH + 1/R_thermal` = $0.5 + 0.5 = 1.0$ | 總散熱係數 |
| `tau = 100 / 1.0 = 100` | 時間常數 $\tau = 100$ 秒 |
| `Gain = 1 / 1.0 = 1.0` | 穩態增益 $K = 1.0$ |
| `[Gain]` = `[1]` | 分子 = $K$ |
| `[tau 1]` = `[100 1]` | 分母 = $100s + 1 = \tau s + 1$ |

> 注意 `[tau 1]` 代表 $\tau s^1 + 1 \cdot s^0$，也就是 $\tau s + 1$。
> 這是一階，所以向量只有 **2 個元素**（最高次 = 1）。

In [ ]:
% === 溫控系統 ===
C_thermal = 100;   % 熱容量 (J/°C)
R_thermal = 2;     % 熱阻 (°C/W)
VH = 0.5;          % 流量×比熱 (W/°C)

% 計算時間常數與穩態增益
% VH + 1/R = 0.5 + 0.5 = 1.0
tau  = C_thermal / (VH + 1/R_thermal);  % = 100/1 = 100 秒
Gain = 1 / (VH + 1/R_thermal);          % = 1/1 = 1.0

% tf([K], [tau 1]) → K / (tau*s + 1)
Gp_temp = tf([Gain], [tau 1]);

disp('=== 溫控轉移函數 ===');
Gp_temp

---

## 五、階躍響應 (Step Response) 視覺比較

### `step()` 函數介紹

`step(sys)` 的意思是：在時間 $t = 0$ 瞬間，給系統一個 **從 0 跳到 1** 的輸入（單位階躍），
然後觀察輸出隨時間如何變化。

這是控制工程中最基本的測試方式，用來觀察系統的：
- **穩定性**：輸出是否會收斂？
- **速度**：收斂要多久？
- **超越量**：有沒有衝過頭？

---

### 1. 衛星系統

In [ ]:
figure(1);
step(Gp_sat, 'r');
title('1. 衛星系統 (紅) - 拋物線發散（不穩定）');
grid on;

### 2. 直流馬達系統

In [ ]:
figure(2);
step(sys_motor, 'g');
title('2. 馬達系統 (綠) - 等速直線增加');
grid on;

### 3. SMIB 發電機

In [ ]:
figure(3);
step(Gp_smib, 'b');
title('3. SMIB 發電機 (藍) - 震盪後穩定');
grid on;

### 4. 溫控水槽

In [ ]:
figure(4);
step(Gp_temp, 'm');
title('4. 溫控水槽 (紫) - 慢速平滑爬坡');
grid on;

---

### 5. 四系統疊加比較圖

In [ ]:
% 統一觀察前 50 秒的階躍響應
figure('Name', '四系統疊加比較圖');
t = 0:0.1:50;

% [y, t_out] = step(sys, t) 會回傳輸出向量 y 和時間向量 t_out
[y_sat, t_sat]     = step(Gp_sat, t);
[y_motor, t_motor] = step(sys_motor, t);
[y_smib, t_smib]   = step(Gp_smib, t);
[y_temp, t_temp]   = step(Gp_temp, t);

plot(t_sat, y_sat, 'r', 'LineWidth', 1.5); hold on;
plot(t_motor, y_motor, 'g', 'LineWidth', 1.5);
plot(t_smib, y_smib, 'b', 'LineWidth', 1.5);
plot(t_temp, y_temp, 'm', 'LineWidth', 1.5);
grid on;

title('四個系統的階躍響應疊加圖 (前 50 秒)');
xlabel('時間 (sec)');
ylabel('系統輸出');
legend('1. 衛星 (紅) - 拋物線失控', ...
       '2. 馬達 (綠) - 直線失控', ...
       '3. 發電機 (藍) - 震盪後穩定', ...
       '4. 溫控水槽 (紫) - 慢速平滑爬坡', ...
       'Location', 'northwest');

---

## 六、四系統特性總結

| 系統 | 轉移函數 | 階數 | 穩定性 | 階躍響應特徵 | 建模方式 |
|------|----------|------|--------|------------|----------|
| 衛星 | $\frac{1}{Js^2}$ | 2 | ❌ 不穩定 | 拋物線發散 | `tf([1],[J 0 0])` |
| 馬達 | $\frac{K_T/JR_a}{s(s+a)}$ | 2 | ❌ 臨界穩定 | 等速直線增加 | `ss(A,B,C,D)` |
| SMIB | $\frac{k}{Ms^2+ds+k}$ | 2 | ✅ 穩定 | 震盪收斂 | `tf([k],[M d k])` |
| 溫控 | $\frac{K}{\tau s+1}$ | 1 | ✅ 穩定 | 指數爬升 | `tf([K],[tau 1])` |

### 🔑 判斷穩定性的快速口訣

看轉移函數的 **分母多項式**：
1. 所有係數都是正的嗎？→ 如果有缺項或零，可能不穩定
2. 分母有沒有純 $s$ 因子（$s$ 的零極點）？→ 有的話輸出會持續增大
3. 極點（分母 = 0 的根）都在左半平面嗎？→ 是就穩定

---

## 七、`[向量]` 寫法的快速對照表

| 你看到的寫法 | 代表的多項式 | 最高次數 |
|-------------|-------------|----------|
| `[1]` | $1$（常數） | 0 次 |
| `[1 2]` | $s + 2$ | 1 次 |
| `[1 0]` | $s$ | 1 次 |
| `[1 0 0]` | $s^2$ | 2 次 |
| `[J 0 0]` | $Js^2$ | 2 次 |
| `[M d k]` | $Ms^2 + ds + k$ | 2 次 |
| `[tau 1]` | $\tau s + 1$ | 1 次 |
| `[1 3 2]` | $s^2 + 3s + 2$ | 2 次 |

> **規則**：向量有 $n$ 個元素 → 多項式最高次 = $n - 1$。
> 第 $i$ 個元素 = $s^{n-i}$ 的係數（從高次排到低次）。